# TradeStation API Usage Examples

This notebook demonstrates how to use the TradeStation API library.

## Prerequisites
- Create a `.env` file with your credentials (see `.env.example`)
- Install dependencies: `pip install -r requirements.txt`

## Environment Selection
Set the environment below:
- `'sim'` - Paper trading (simulation) - **SAFE**
- `'prod'` - Live trading - **REAL MONEY** ⚠️


In [6]:
# ============================================================
# CONFIGURATION - Set environment here
# ============================================================

# Choose environment: 'sim' or 'prod'
ENVIRONMENT = 'sim'  # CHANGE THIS to 'prod' for live trading (REAL MONEY)

print(f"Environment: {ENVIRONMENT}")
print(f"⚠️  {'SIMULATION MODE - Paper trading' if ENVIRONMENT == 'sim' else 'PRODUCTION MODE - REAL MONEY'}")


Environment: sim
⚠️  SIMULATION MODE - Paper trading


In [7]:
# ============================================================
# SETUP - Import libraries and initialize API
# ============================================================

# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

# Add the parent directory to path so we can import the api module
sys.path.insert(0, str(Path().absolute().parent))

# Import the TradeStation API
from tradestation.api import TradeStationAPI

# Initialize the API with the selected environment
# This will load credentials from .env and set the appropriate API URL
api = TradeStationAPI(ENVIRONMENT)

# Get the account ID from configuration
account_id = api.config.account_id

print(f"\n✓ API initialized successfully")
print(f"✓ Base URL: {api.config.base_url}")
print(f"✓ Account ID: {account_id}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Loaded configuration from: .env
✓ Environment: sim

✓ API initialized successfully
✓ Base URL: https://sim-api.tradestation.com
✓ Account ID: SIM2977785M


---
## Market Data

Get historical and real-time market data.


In [ ]:
# ============================================================
# Get Bar Chart Data (Historical Prices)
# ============================================================

# Get last 10 bars of 5-minute data for ES (S&P 500)
symbol = 'ES'
bars = api.market_data.get_bars(
    symbol=symbol,
    interval=5,          # 5-minute intervals
    unit='Minute',       # Unit of time
    bars_back=10         # Number of bars to retrieve
)

print(f"Retrieved {len(bars.get('Bars', []))} bars for {symbol}")
print(f"\nFirst bar:")
print(json.dumps(bars.get('Bars', [])[0] if bars.get('Bars') else {}, indent=2))
print(f"\nLast bar:")
print(json.dumps(bars.get('Bars', [])[-1] if bars.get('Bars') else {}, indent=2))


Retrieved 10 bars for ES

First bar:
{
  "High": "66.98",
  "Low": "66.92",
  "Open": "66.92",
  "Close": "66.97",
  "TimeStamp": "2026-01-05T20:15:00Z",
  "TotalVolume": "5504",
  "DownTicks": 19,
  "DownVolume": 2605,
  "OpenInterest": "0",
  "IsRealtime": false,
  "IsEndOfHistory": false,
  "TotalTicks": 47,
  "UnchangedTicks": 0,
  "UnchangedVolume": 0,
  "UpTicks": 28,
  "UpVolume": 2899,
  "Epoch": 1767644100000,
  "BarStatus": "Closed"
}

Last bar:
{
  "High": "67.2",
  "Low": "67.02",
  "Open": "67.13",
  "Close": "67.06",
  "TimeStamp": "2026-01-05T21:00:00Z",
  "TotalVolume": "137216",
  "DownTicks": 403,
  "DownVolume": 66338,
  "OpenInterest": "0",
  "IsRealtime": false,
  "IsEndOfHistory": true,
  "TotalTicks": 806,
  "UnchangedTicks": 0,
  "UnchangedVolume": 0,
  "UpTicks": 403,
  "UpVolume": 70878,
  "Epoch": 1767646800000,
  "BarStatus": "Closed"
}


In [9]:
# ============================================================
# Get Symbol Details
# ============================================================

# Get detailed information about a symbol
symbol = 'ES'
details = api.market_data.get_symbol_details(symbol)

print(f"Symbol Details for {symbol}:")
print(json.dumps(details, indent=2))


Symbol Details for ES:
{
  "Symbols": [
    {
      "AssetType": "STOCK",
      "Country": "United States",
      "Currency": "USD",
      "Description": "Eversource Energy",
      "Exchange": "NYSE",
      "Symbol": "ES",
      "Root": "ES",
      "PriceFormat": {
        "Format": "Decimal",
        "Decimals": "2",
        "IncrementStyle": "Simple",
        "Increment": "0.01",
        "PointValue": "1"
      },
      "QuantityFormat": {
        "Format": "Decimal",
        "Decimals": "0",
        "IncrementStyle": "Simple",
        "Increment": "1",
        "MinimumTradeQuantity": "1"
      }
    }
  ],
  "Errors": []
}


---
## Account Information

Retrieve account balances and positions.


In [14]:
# ============================================================
# Get Account Balances
# ============================================================

# Get real-time account balances
balances = api.account.get_balances(account_id)

print(f"Account Balances for {account_id}:")
print(json.dumps(balances, indent=2))

# Extract key metrics
# Note: TradeStation API may return numeric values as strings, so we convert them
if 'Balances' in balances and len(balances['Balances']) > 0:
    balance = balances['Balances'][0]
    print(f"\nKey Metrics:")
    
    # Helper function to safely convert to float for formatting
    def to_float(value, default=0.0):
        try:
            return float(value) if value is not None else default
        except (ValueError, TypeError):
            return default
    
    account_value = to_float(balance.get('AccountValue', 0))
    cash_balance = to_float(balance.get('CashBalance', 0))
    buying_power = to_float(balance.get('BuyingPower', 0))
    
    print(f"  Account Value: ${account_value:,.2f}")
    print(f"  Cash Balance: ${cash_balance:,.2f}")
    print(f"  Buying Power: ${buying_power:,.2f}")


Account Balances for SIM2977785M:
{
  "Balances": [
    {
      "AccountID": "SIM2977785M",
      "AccountType": "Margin",
      "CashBalance": "1000000",
      "BuyingPower": "4000000",
      "Equity": "1000000",
      "MarketValue": "0",
      "TodaysProfitLoss": "0",
      "UnclearedDeposit": "0",
      "BalanceDetail": {
        "CostOfPositions": "0",
        "DayTrades": "0",
        "MaintenanceRate": "0",
        "OptionBuyingPower": "1000000",
        "OptionsMarketValue": "0",
        "OvernightBuyingPower": "2000000",
        "RequiredMargin": "0",
        "UnsettledFunds": "0",
        "DayTradeExcess": "1000000",
        "RealizedProfitLoss": "0",
        "UnrealizedProfitLoss": "0"
      },
      "Commission": "0"
    }
  ],
  "Errors": []
}

Key Metrics:
  Account Value: $0.00
  Cash Balance: $1,000,000.00
  Buying Power: $4,000,000.00


In [15]:
# ============================================================
# Get Current Positions
# ============================================================

# Get all current positions
positions = api.account.get_positions(account_id)

print(f"Current Positions for {account_id}:")

# Helper function to safely convert to float for formatting
def to_float(value, default=0.0):
    try:
        return float(value) if value is not None else default
    except (ValueError, TypeError):
        return default

if 'Positions' in positions and len(positions['Positions']) > 0:
    print(f"\nFound {len(positions['Positions'])} position(s):\n")
    
    total_unrealized_pnl = 0
    
    for pos in positions['Positions']:
        symbol = pos.get('Symbol', 'N/A')
        quantity = to_float(pos.get('Quantity', 0))
        avg_price = to_float(pos.get('AveragePrice', 0))
        last_price = to_float(pos.get('Last', 0))
        unrealized_pnl = to_float(pos.get('UnrealizedProfitLoss', 0))
        
        print(f"  {symbol}:")
        print(f"    Quantity: {quantity:.0f}")
        print(f"    Avg Price: ${avg_price:.2f}")
        print(f"    Last Price: ${last_price:.2f}")
        print(f"    Unrealized P&L: ${unrealized_pnl:.2f}")
        print()
        
        total_unrealized_pnl += unrealized_pnl
    
    print(f"Total Unrealized P&L: ${total_unrealized_pnl:.2f}")
else:
    print("  No positions found.")


Current Positions for SIM2977785M:
  No positions found.


---
## Order Management

Confirm, place, and manage orders.

⚠️ **CAUTION**: Even in simulation mode, always double-check order details!


In [17]:
# ============================================================
# Confirm Order (Dry-run - Does NOT place the order)
# ============================================================

# This validates the order and shows estimated costs/margins
# It does NOT actually place the order - safe to run

# Note: Using a stock symbol (AAPL) which works with most simulation accounts
# If your account supports futures, you can try 'ESZ24' or '@ES' instead

confirmation = api.orders.confirm_order(
    account_id=account_id,
    symbol='AAPL',         # Apple stock (use a stock your account supports)
    quantity=1,            # 1 share
    action='BUY',          # BUY or SELL
    order_type='Market',   # Market order
    time_in_force='Day'    # Day order (expires end of day)
)

print("Order Confirmation (NOT placed):")
print(json.dumps(confirmation, indent=2))

# Extract key information
# Helper function to safely convert to float
def to_float(value, default=0.0):
    try:
        return float(value) if value is not None else default
    except (ValueError, TypeError):
        return default

if 'Route' in confirmation:
    print(f"\nOrder Details:")
    print(f"  Route: {confirmation.get('Route', 'N/A')}")
    
    estimated_cost = to_float(confirmation.get('EstimatedCost', 0))
    print(f"  Estimated Cost: ${estimated_cost:,.2f}")
    
    # Initial margin is for futures - may not be present for stocks
    if 'InitialMarginRequirement' in confirmation:
        margin = to_float(confirmation.get('InitialMarginRequirement', 0))
        print(f"  Initial Margin: ${margin:,.2f}")


Order Confirmation (NOT placed):
{
  "Confirmations": [
    {
      "OrderAssetCategory": "EQUITY",
      "Currency": "USD",
      "Route": "Intelligent",
      "TimeInForce": {
        "Duration": "DAY"
      },
      "AccountID": "SIM2977785M",
      "OrderConfirmID": "jzp4drJnn0iokq4ntZvPeQ",
      "EstimatedPrice": "267.27",
      "EstimatedCost": "267.27",
      "DebitCreditEstimatedCost": "267.27",
      "EstimatedCommission": "1",
      "SummaryMessage": "Buy 1 AAPL @ Market"
    }
  ]
}


In [18]:
# ============================================================
# Get Current Orders
# ============================================================

# Retrieve all current (active) orders
orders = api.orders.get_orders(account_id)

print(f"Current Orders for {account_id}:")

if 'Orders' in orders and len(orders['Orders']) > 0:
    print(f"\nFound {len(orders['Orders'])} order(s):\n")
    
    for order in orders['Orders']:
        order_id = order.get('OrderID', 'N/A')
        symbol = order.get('Legs', [{}])[0].get('Symbol', 'N/A')
        action = order.get('Legs', [{}])[0].get('BuyOrSell', 'N/A')
        quantity = order.get('Legs', [{}])[0].get('QuantityOrdered', 0)
        status = order.get('Status', 'N/A')
        
        print(f"  Order ID: {order_id}")
        print(f"    Symbol: {symbol}")
        print(f"    Action: {action}")
        print(f"    Quantity: {quantity}")
        print(f"    Status: {status}")
        print()
else:
    print("  No active orders found.")


Current Orders for SIM2977785M:
  No active orders found.


---
## Streaming Data (Optional)

**Note**: Streaming endpoints keep the connection open and continuously push data.
They work well in scripts but can be tricky in notebooks.

Example usage pattern:


In [ ]:
# ============================================================
# Streaming Example (Commented out - use in a script instead)
# ============================================================

# Streaming is better suited for standalone scripts
# Uncomment and run in a Python script for live streaming:

# Stream real-time bar data
for bar_data in api.market_data.stream_bars(
    symbol='@ES',
    interval=1,
    unit='Minute'
):
    print(f"New bar: {bar_data}")
    # Process each bar as it arrives
    # Press Ctrl+C to stop

# Stream real-time positions
for position_update in api.account.stream_positions(account_id):
    print(f"Position update: {position_update}")
    # Process position updates

# Stream real-time order updates
for order_update in api.orders.stream_orders(account_id):
    print(f"Order update: {order_update}")
    # Process order status changes

print("Streaming examples are commented out.")
print("See the code cell above for usage patterns.")
print("Run streaming code in a Python script for best results.")


Streaming examples are commented out.
See the code cell above for usage patterns.
Run streaming code in a Python script for best results.


---
## Additional Resources

- **Library Documentation**: See `api/README.md` and `api/USAGE.md`
- **Code Examples**: See `api/example.py` for standalone Python examples
- **API Specification**: See `api_spec/openapi.json` for TradeStation API details
- **Original Demo**: See `ts_api_demo.ipynb` for raw API call examples

---

**Happy Trading! 📈**
